# Notebook zur erstellung des Datasets

In [50]:
import os
import random
import json
import pandas as pd

### Parameter

In [51]:
DATA_DIR = "../sensor_and_video_data"
LABELED_DIR = "../Labeled"
SEED = 42
# Files expeted in a run folder for it to be considered complete
EXPECTED_FILES = ["selected_peaks", "synch_data", "Timestamps"]

CONVERSION_FACTOR = 2.4  # To convert from video frames to sensor samples


# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15


### Helper functions

In [52]:
def check_run_complete(folder, keywords):
    files = os.listdir(os.path.join(DATA_DIR, folder))

    for word in keywords:
        if not any(word in file for file in files):
            return False
    return True

### Runweise Aufteilung in Train/Validation/Test + auschließen von unvollständigen Runs

In [53]:

# Collect all run directories
runs = [
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]

# Exclude incomplete runs
for run in runs[:]:
    if not check_run_complete(run, EXPECTED_FILES):
        runs.remove(run)
        print(f"Removed incomplete run: {run}")

print(f"Found {len(runs)} runs.")

# Shuffle runs for randomness
random.seed(SEED)
random.shuffle(runs)

# Split into train, val, test
num_runs = len(runs)
train_split = int(TRAIN_SPLIT * num_runs)
val_split = int((TRAIN_SPLIT + VAL_SPLIT) * num_runs)
train_runs = runs[:train_split]
val_runs = runs[train_split:val_split]
test_runs = runs[val_split:]

print(train_runs)
print(val_runs)
print(test_runs)

Found 16 runs.
['0721_0853_noheadsensor', '0724_0723', '0720_0828', '0720_0858', '0802_0800', '0725_0709', '0727_0812', '0721_0928_noheadsensor', '0713_0903', '0713_0932', '0801_0758']
['0803_0734', '0720_0748']
['0727_0735', '0713_0833', '0717_0717']


### Funktion zur Erstellen des Label Vektors

One Hot-Encoding. Bezieht sich auf Video Frames und wird nur Umgerechnet !!!

In [54]:
def create_label_vector(run_file, len_of_run):
    file_path = os.path.join(LABELED_DIR, run_file)
    if os.path.isfile(file_path):
        print(f"Label file exists for {run_file}")
        label_vector = ["nothing"] * len_of_run

        with open(file_path, 'r') as f:
            data = json.load(f)

        raw = data["button_presses"]
        # *2.4 to convert from video frames to sensor samples
        sequence_width_half = int(int(data["sequence_width"])*CONVERSION_FACTOR) // 2

        button_presses = []
        for entry in raw.split(";"):
            entry = entry.strip()
            if not entry:
                continue
            label, idx = entry.split(":")
            button_presses.append((int(float(idx.strip()) * CONVERSION_FACTOR), label.strip())) # convert from video frames to sensor samples

        for idx, label in button_presses:
            if 0 <= idx < len_of_run:
                label_vector[idx-sequence_width_half : idx + sequence_width_half + 1] = [label] * (2 * sequence_width_half + 1)
        one_hot_labels = pd.get_dummies(label_vector)
        return one_hot_labels
        
    else:
        print(f"Label file missing for {run_file}")


create_label_vector("0717_0717_sequences.json", 5000)
    

Label file exists for 0717_0717_sequences.json


,Jumping,nothing
0,False,True
1,False,True
2,False,True
3,False,True
4,False,True
...,...,...
4995,False,True
4996,False,True
4997,False,True
4998,False,True


### Zusammenfüge des Datasets

Synchronisation des Videos und der Sensordaten durch die Dateien Timestamps.json und selected_peaks.json. In selected_peaks.json sind die PacketCounter werte zu finden and denen die verschiedenen Sensoren geklopft wurden. In Timestamps.json sind die dazugehörigen Video Frames. Start des Sliding Windows nach klopfen des letzten Sensors

In [ ]:
def synch_data(run_file):
    file_path_synch = os.path.join(DATA_DIR, run_file, ".json")
    file_path_selected_peaks = os.path.join(DATA_DIR, run_file, "selected_peaks.json")

    # Load synch data
    if os.path.isfile(file_path_synch):
        print(f"Synch file exists for {run_file}")

        with open(file_path_synch, 'r') as f:
            data_synch = json.load(f)

    # Load selected peaks data
    if os.path.isfile(file_path_selected_peaks):
        print(f"Selected peaks file exists for {run_file}")

        with open(file_path_selected_peaks, 'r') as f:
            data_selected_peaks = json.load(f)

    
